# Baseline Model



In [1]:
# imports
import numpy as np
import librosa
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

C:\Users\lilyc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# login Hugging Face Hub
import os
from dotenv import load_dotenv

load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN) 

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from datasets import load_dataset
import os

# ds = load_dataset("SleepyJesse/ai_music_tiny", split="train", cache_dir="./data")
ds = load_dataset("SleepyJesse/ai_music_tiny")
print(f"Dataset loaded with {len(ds)} samples")

Generating train split: 100%|██████████| 1000/1000 [00:31<00:00, 31.62 examples/s]


Dataset loaded with 1 samples


Sampling and data processing

In [4]:
def sample_per_class(dataset, label_key="label", samples_per_class=500):
    class_counts = {}
    selected = []

    for example in dataset:
        label = example[label_key]

        if label not in class_counts:
            class_counts[label] = 0

        if class_counts[label] < samples_per_class:
            selected.append(example)
            class_counts[label] += 1

        if all(v >= samples_per_class for v in class_counts.values()):
            break

    return selected

In [5]:
def segment_audio(audio_array, sr, clip_duration=5):
    clip_length = sr * clip_duration
    segments = []

    for start in range(0, len(audio_array), clip_length):
        end = start + clip_length
        segment = audio_array[start:end]

        if len(segment) < clip_length:
            break

        segments.append(segment)

    return segments

Extract mel frequency cepstral coefficients

In [6]:
def extract_mfcc_features(audio, sr, n_mfcc=20):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc
    )

    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])

Build dataset

In [7]:
def build_feature_dataset(dataset_subset):

    X = []
    y = []

    for example in dataset_subset:

        audio = example["audio"]["array"]
        sr = example["audio"]["sampling_rate"]
        label = example["label"]

        segments = segment_audio(audio, sr)

        for seg in segments:
            features = extract_mfcc_features(seg, sr)

            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

Train model

In [8]:
subset = sample_per_class(ds["train"], samples_per_class=500)

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

In [ ]:
X, y = build_feature_dataset(subset)

print("Feature shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = LogisticRegression(max_iter=2000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))